# 03 Features

**What this notebook does.** It builds the development subset and the time split, adds calendar, oil, lag and rolling features, replaces the leaky promotion columns, encodes and scales the features and writes the feature list to `feature_spec.json`. A correlation check at the end removes one duplicated column from the linear feature list. The cells depend on each other and must be run from top to bottom.

## 1. Development subset and time split

**Purpose.** The full sales grid has 194.7M rows, which is too heavy for iterative feature work. This step builds a small, reproducible subset and fixes the train / validation / test split before any feature is created.

**Method.**
- **Stores:** the 3 states with the most stores open since the start of the data, up to 4 stores per state (`random_state=42`). Azuay has only 2 such stores, so the subset has 10 stores (Pichincha 4, Guayas 4, Azuay 2).
- **Items:** 15% of the items of each family. Three families (baby care, books, home appliances) have no rows in the subset, probably because 15% of their few items rounds to zero, so the subset has 30 of the 33 families.
- Only the matching rows are read from `clean/sales_grid` (partition filter) instead of loading the full grid.
- `family`, `perishable` and the `is_closed` flag (no-record store-days, see notebook 02) are attached.
- **Time split:** last 16 days = test, the 16 days before = validation, everything earlier = train. The split is chronological, never random, to avoid leakage.

**Output.** `features/subset_base.parquet` and `features/subset_definition.json` (seed, store list, item list, cut dates), so the subset can be rebuilt exactly.

**Findings.** The subset has 5,888,022 rows (3.0% of the grid), 10 stores and 607 items. Train covers 2013-01-02 to 2017-07-14 (1,651 days, 5,732,606 rows), validation 2017-07-15 to 2017-07-30 (16 days, 77,601 rows) and test 2017-07-31 to 2017-08-15 (16 days, 77,815 rows). 52,605 rows belong to no-record store-days, all in the train period.

**Limitation.** The subset favours large provinces: only three provinces are included, Azuay has two stores and Manabí is not represented. Results describe this subset, not the whole chain. The baselines are repeated on all 54 stores; the full pipeline on all data belongs to the next semester.

In [1]:
import json
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
DIM = BASE / "clean" / "dim"
(BASE / "features").mkdir(exist_ok=True)

stores = pd.read_parquet(DIM / "stores.parquet")
items = pd.read_parquet(DIM / "items.parquet")
cal_flags = pd.read_parquet(DIM / "calendar_store.parquet",
                            columns=["date", "store_nbr", "store_open_date", "is_closed"])

opened = cal_flags[["store_nbr", "store_open_date"]].drop_duplicates()
eligible = stores.merge(opened, on="store_nbr")
eligible = eligible[eligible["store_open_date"] <= "2013-01-02"]

top_states = eligible["state"].value_counts().index[:3]
sub_stores = (eligible[eligible["state"].isin(top_states)]
              .sample(frac=1, random_state=42)
              .groupby("state", observed=True).head(4))
sub_items = items.groupby("family", observed=True).sample(frac=0.15, random_state=42)

store_list = sub_stores["store_nbr"].astype(int).tolist()
item_list = sub_items["item_nbr"].astype(int).tolist()

dataset = ds.dataset(BASE / "clean" / "sales_grid", format="parquet", partitioning="hive")
flt = ds.field("store_nbr").isin(store_list) & ds.field("item_nbr").isin(item_list)
df = dataset.to_table(filter=flt).to_pandas()
df = (df.drop(columns="year", errors="ignore")
        .sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True))

df = df.merge(items[["item_nbr", "family", "perishable"]], on="item_nbr", how="left")
df = df.merge(cal_flags[["date", "store_nbr", "is_closed"]], on=["date", "store_nbr"], how="left")
df["is_closed"] = df["is_closed"].eq(True)
print("closed-day zero rows in subset:", f"{int(df['is_closed'].sum()):,}")

H = 16
days = np.sort(df["date"].unique())
test_start, val_start = days[-H], days[-2 * H]
df["split"] = pd.Categorical(
    np.select([df["date"] >= test_start, df["date"] >= val_start],
              ["test", "val"], default="train"),
    categories=["train", "val", "test"])

print(f"\nrows: {len(df):,} ({len(df) / 194_660_451:.1%} of full grid)"
      f" | stores: {len(store_list)} | items: {len(item_list)}")
print(sub_stores[["store_nbr", "city", "state", "type"]].to_string(index=False))
print()
print(df.groupby("split", observed=True)["date"].agg(["min", "max", "nunique", "size"]))

df.to_parquet(BASE / "features" / "subset_base.parquet", index=False)
with open(BASE / "features" / "subset_definition.json", "w") as f:
    json.dump({"seed": 42, "stores": store_list, "items": item_list, "horizon_days": H,
               "val_start": str(pd.Timestamp(val_start).date()),
               "test_start": str(pd.Timestamp(test_start).date())}, f, indent=2)
print("\nsaved subset_base.parquet and subset_definition.json")

closed-day zero rows in subset: 52,605

rows: 5,888,022 (3.0% of full grid) | stores: 10 | items: 607
 store_nbr      city     state type
        49     Quito Pichincha    A
        30 Guayaquil    Guayas    C
        24 Guayaquil    Guayas    D
        44     Quito Pichincha    A
        10     Quito Pichincha    C
        11   Cayambe Pichincha    B
        39    Cuenca     Azuay    B
        26 Guayaquil    Guayas    D
        28 Guayaquil    Guayas    E
        37    Cuenca     Azuay    D

             min        max  nunique     size
split                                        
train 2013-01-02 2017-07-14     1651  5732606
val   2017-07-15 2017-07-30       16    77601
test  2017-07-31 2017-08-15       16    77815

saved subset_base.parquet and subset_definition.json


## 2. Calendar, store and oil features

**Purpose.** Attach context that explains demand (holidays, paydays, store attributes, oil price) to the subset, and flag extreme sales values, without leaking information from the validation or test period.

**Method.**
- Calendar and store attributes come from `calendar_store` (one row per date × store), so they are joined once instead of being repeated in the 194.7M-row grid.
- `transactions` is deliberately left out: it is only known after the day has happened, so using it in a 16-day-ahead model would be leakage.
- `earthquake` and `is_event_other` are joined too, but `earthquake` cannot be known in advance and is used for analysis only (see section 5).
- Oil price is shifted by 16 days (`oil_lag16`): at forecast time the price of the target day is not known. The first oil value was back-filled in the source and is left missing here.
- `is_extreme` marks sales above the 99.9th percentile of each family, computed on all train rows of the family (including zero-filled days). Rows are flagged, not deleted, because the data does not show whether they are errors or genuine bulk orders.
- The number of rows is checked before and after the joins.

**Output.** `features/subset_step1.parquet`, plus counts of missing values, no-record rows and extreme rows per split.

**Findings.** `oil_lag16` is missing for 27,085 rows (the first 16 days of the subset, 2 to 17 January 2013). Extreme rows are 0.098% of train, 0.108% of validation and 0.104% of test, so the validation and test periods contain no new kind of extreme value. Three families (baby care, books, home appliances) have no threshold because they are absent from the subset.

**Limitation.** No-record store-days (rows whose zeros are not demand) are kept and flagged here; they are set to missing in the lag features and excluded from training and metrics.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
DIM = BASE / "clean" / "dim"
H = 16  # forecast horizon (days)

df = pd.read_parquet(BASE / "features" / "subset_base.parquet")
n_rows = len(df)
print(f"loaded: {n_rows:,} rows")

print("\nclosed-day rows by split:")
print(df.groupby("split", observed=True)["is_closed"].agg(rows="sum", share="mean").round(4))

# calendar and store attributes (one row per date x store)
cal_cols = ["date", "store_nbr", "city", "state", "store_type", "cluster",
            "hol_national", "hol_local", "is_holiday", "is_workday_swap",
            "is_event_other", "earthquake", "is_payday", "dow", "month"]
cal = pd.read_parquet(DIM / "calendar_store.parquet", columns=cal_cols)
df = df.merge(cal, on=["date", "store_nbr"], how="left")

# oil price as known H days earlier
oil = pd.read_parquet(DIM / "oil.parquet").set_index("date")["oil_price"]
oil.iloc[0] = np.nan   # the first value was back-filled from the next day; leave it missing
all_days = pd.date_range(oil.index.min(), oil.index.max())
oil_lag = (oil.reindex(all_days).ffill().shift(H)
              .rename(f"oil_lag{H}").rename_axis("date").reset_index())
df = df.merge(oil_lag, on="date", how="left")

# extreme-sales flag, threshold from the train split only
fam = df["family"].astype("category")
is_train = df["split"].eq("train")
thr = (df.loc[is_train].groupby(fam[is_train], observed=True)["unit_sales"].quantile(0.999)
         .reindex(fam.cat.categories))
thr_row = thr.to_numpy(dtype="float32")[fam.cat.codes.to_numpy()]
df["is_extreme"] = df["unit_sales"].to_numpy() > thr_row

assert len(df) == n_rows, "merge changed the number of rows"

new_cols = [c for c in df.columns if c in cal_cols[2:] or c in (f"oil_lag{H}", "is_extreme")]
miss = df[new_cols].isna().sum()
print("\nmissing values in new columns:")
print(miss[miss > 0].to_string() if (miss > 0).any() else "none")
print("\nextreme rows by split:")
print(df.groupby("split", observed=True)["is_extreme"].agg(rows="sum", share="mean").round(5))
print("\np99.9 thresholds (train only):")
print(thr.round(1).to_string())

df.to_parquet(BASE / "features" / "subset_step1.parquet", index=False)
print(f"\nsaved subset_step1.parquet: {df.shape[0]:,} rows x {df.shape[1]} cols")

loaded: 5,888,022 rows

closed-day rows by split:
        rows   share
split               
train  52605  0.0092
val        0  0.0000
test       0  0.0000

missing values in new columns:
oil_lag16    27085

extreme rows by split:
       rows    share
split               
train  5646  0.00098
val      84  0.00108
test     81  0.00104

p99.9 thresholds (train only):
AUTOMOTIVE                      7.0
BABY CARE                       NaN
BEAUTY                          7.0
BEVERAGES                     305.9
BOOKS                           NaN
BREAD/BAKERY                   78.6
CELEBRATION                    15.0
CLEANING                       59.0
DAIRY                          83.0
DELI                           94.0
EGGS                          123.0
FROZEN FOODS                  244.2
GROCERY I                     165.0
GROCERY II                     51.0
HARDWARE                        8.6
HOME AND KITCHEN I             34.0
HOME AND KITCHEN II            13.0
HOME APPLIANCES      

## 3. Lag and rolling features

**Purpose.** Give the models the recent history of each store-item series (level, weekly and yearly pattern, intermittency) in a form that never uses information from the days being forecast.

**Method.**
- The forecast horizon is 16 days, so **every lag is at least 16 days** (16, 21, 28, 364). A shorter lag (e.g. yesterday) would contain the answer for validation and test rows.
- Sales are arranged in a day × series matrix on the true calendar, so a lag of 28 means exactly 28 calendar days, even across the missing 25 December dates.
- No-record days are set to missing before any lag or average is computed, so they do not pull averages down.
- `roll_mean_7`, `roll_mean_28` and `zero_share_28` (share of zero-sales days, an intermittency signal) are computed on the series shifted by 16 days. A window needs at least half of its days to produce a value.
- `series_age` (days since the first sales record) is kept for analysis only. It is not a model feature: the grid starts at the first sale, so a row with age 0 always has a sale.
- A check on one series compares each lag with a manual calculation.

**Output.** `features/subset_features.parquet` and a table of missing values per split.

**Findings.** The lag check passes. Missing values are low in validation and test (`lag_16` 0.3%, `lag_28` 0.4% to 0.5%), mostly products with no history, and higher in train (`lag_16` 2.5%). `lag_364` is missing for 30.2% of train rows (first year) and for about 11% of validation and test rows (products launched less than a year earlier).

**Limitation.** Validation and test rows use earlier actuals as history, which is allowed because those days are known at the forecast origin (each target day has its own origin, 16 days earlier). Items whose first sale falls inside the validation or test period have no history, so their lags stay missing (cold start).

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
H = 16                    # forecast horizon (days)
LAGS = [16, 21, 28, 364]  # every lag >= H
WINDOWS = [7, 28]

df = pd.read_parquet(BASE / "features" / "subset_step1.parquet")
df = df.sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True)
n_rows = len(df)

# day x series matrix on the true calendar
all_days = pd.date_range(df["date"].min(), df["date"].max())
d_idx = all_days.get_indexer(df["date"])
key = df["store_nbr"].astype("int64") * 10_000_000 + df["item_nbr"].astype("int64")
s_codes, s_uniq = pd.factorize(key, sort=True)

obs = df["unit_sales"].to_numpy(dtype="float32").copy()
obs[df["is_closed"].to_numpy()] = np.nan          # no-record days are not demand
Y = np.full((len(all_days), len(s_uniq)), np.nan, dtype="float32")
Y[d_idx, s_codes] = obs
Y = pd.DataFrame(Y)
Yh = Y.shift(H)                                    # history known at forecast origin

for k in LAGS:
    df[f"lag_{k}"] = Y.shift(k).to_numpy()[d_idx, s_codes]
for w in WINDOWS:
    m = Yh.rolling(w, min_periods=(w + 1) // 2).mean()
    df[f"roll_mean_{w}"] = m.to_numpy(dtype="float32")[d_idx, s_codes]
zero = (Yh == 0).astype("float32").where(Yh.notna())
df["zero_share_28"] = zero.rolling(28, min_periods=14).mean().to_numpy(dtype="float32")[d_idx, s_codes]

first_idx = pd.Series(d_idx).groupby(s_codes).transform("min").to_numpy()
df["series_age"] = (d_idx - first_idx).astype("int16")   # days since the first sales record (analysis only)

assert len(df) == n_rows

# sanity check on one series: lag must equal the sales k days earlier
s0 = s_uniq[len(s_uniq) // 2]
sub = df[key.to_numpy() == s0].set_index("date")
ser = sub["unit_sales"].where(~sub["is_closed"])
for k in LAGS:
    exp = ser.reindex(ser.index - pd.Timedelta(days=k)).to_numpy()
    assert np.allclose(exp, sub[f"lag_{k}"].to_numpy(), equal_nan=True), f"lag_{k} mismatch"
print("lag check on one series: OK")

feat = [f"lag_{k}" for k in LAGS] + [f"roll_mean_{w}" for w in WINDOWS] + ["zero_share_28"]
print("\nshare of missing values by split:")
print(df.groupby("split", observed=True)[feat].agg(lambda s: s.isna().mean()).round(3).T)
print("families in subset:", df["family"].nunique())

df.to_parquet(BASE / "features" / "subset_features.parquet", index=False)
print(f"\nsaved subset_features.parquet: {df.shape[0]:,} rows x {df.shape[1]} cols")

lag check on one series: OK

share of missing values by split:
split          train    val   test
lag_16         0.025  0.003  0.003
lag_21         0.029  0.004  0.004
lag_28         0.035  0.004  0.005
lag_364        0.302  0.110  0.105
roll_mean_7    0.023  0.004  0.003
roll_mean_28   0.031  0.005  0.006
zero_share_28  0.031  0.005  0.006
families in subset: 30

saved subset_features.parquet: 5,888,022 rows x 34 cols


## 4. Leakage check on promotion and a safe replacement

**Purpose.** `onpromotion` is only recorded on rows where something was sold. The zero rows added when the grid was rebuilt have no promotion value, so "promotion unknown" can reveal "no sale that day". This step proves it and replaces the leaky columns.

**Method.**
- From April 2014 (after the period in which promotion was not recorded) the share of rows with a recorded promotion is compared between rows with `unit_sales > 0` and rows with `unit_sales == 0`.
- `promo` and `promo_known` are removed from the feature table. `onpromotion` stays in the file for exploratory analysis only and must not be a model input.
- Replacement: `promo_rate_observed_28`, the share of promotion days in the 28 days that end 16 days before the target day (only recorded days count). It uses history only.

**Output.** The evidence table, `promo_rate_observed_28` with its missing share per split, and the updated `features/subset_features.parquet`.

**Findings.** From April 2014 the share of rows with a recorded promotion is 0.0 for rows with `unit_sales == 0` and 1.0 for rows with `unit_sales > 0`, so the missingness reveals whether a sale happened. `promo_rate_observed_28` is missing for 46.3% of train rows and 34.6% of validation and test rows, because it needs at least 14 recorded days (days with sales) in the 28-day window; its missing pattern therefore also reflects how often the product sold in the past. It uses only history, so it does not leak.

**Limitation.** In the original competition the promotion flag of future days is supplied, so a target-day promotion feature would be legitimate there. In this rebuilt grid it is not recorded on zero-sales days, so it cannot be used without leaking. Promotion effects are studied in the EDA on rows with sales, with this selection effect stated.

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
H = 16

df = pd.read_parquet(BASE / "features" / "subset_features.parquet")
df = df.sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True)
n_rows = len(df)

# evidence: after the null era, "promotion unknown" means "no sale that day"
late = df[df["date"] >= "2014-04-01"]
print("share of rows with a recorded promotion, from 2014-04 onwards:")
print((late["onpromotion"] >= 0).groupby(late["unit_sales"] > 0).mean().round(3)
          .rename({False: "unit_sales == 0", True: "unit_sales > 0"}))

# safe replacement: promotion share over the 28 days ending H days earlier
all_days = pd.date_range(df["date"].min(), df["date"].max())
d_idx = all_days.get_indexer(df["date"])
key = df["store_nbr"].astype("int64") * 10_000_000 + df["item_nbr"].astype("int64")
s_codes, _ = pd.factorize(key, sort=True)

promo = df["onpromotion"].to_numpy().astype("float32")
promo[promo < 0] = np.nan
P = np.full((len(all_days), s_codes.max() + 1), np.nan, dtype="float32")
P[d_idx, s_codes] = promo
share = pd.DataFrame(P).shift(H).rolling(28, min_periods=14).mean()
df["promo_rate_observed_28"] = share.to_numpy(dtype="float32")[d_idx, s_codes]

df = df.drop(columns=["promo", "promo_known"], errors="ignore")
assert len(df) == n_rows

print("\npromo_rate_observed_28 missing by split:")
print(df.groupby("split", observed=True)["promo_rate_observed_28"].apply(lambda s: s.isna().mean()).round(3))

df.to_parquet(BASE / "features" / "subset_features.parquet", index=False)
print(f"\nsaved subset_features.parquet: {df.shape[0]:,} rows x {df.shape[1]} cols")

share of rows with a recorded promotion, from 2014-04 onwards:
unit_sales
unit_sales == 0    0.0
unit_sales > 0     1.0
Name: onpromotion, dtype: float64

promo_rate_observed_28 missing by split:
split
train    0.463
val      0.346
test     0.346
Name: promo_rate_observed_28, dtype: float64

saved subset_features.parquet: 5,888,022 rows x 35 cols


## 5. Encoding, scaling and the feature list

**Purpose.** Prepare the features for modelling and record exactly which columns a model may use. Everything that is learned from the data (categories, means, standard deviations) is learned on the **train split only**, then applied unchanged to validation and test.

**Method.**
- **Rows used for fitting:** train split, store open (no-record days are not demand). The same rule is stored in the spec as the training filter.
- **Categorical columns** (`family`, `city`, `state`, `store_type`) become integer codes using the categories seen in train; anything unseen would get -1. The codes are labels, not quantities: tree models must be told that `categorical_features` are categorical, and linear models do not receive them (they need one-hot encoding when the model is fitted). `cluster`, `dow` and `month` are also categorical (`dow` and `month` may be encoded cyclically).
- **Scaling:** sales-like columns (lags and rolling means) are transformed with `log1p` because demand is highly skewed, then standardised with train mean and standard deviation. Other numeric columns are standardised directly. Missing values are left missing; they are not filled with statistics from later periods.
- **Leakage control:** the feature lists are explicit and an `assert` stops the cell if the target or a column that depends on the target day (`unit_sales_raw`, `is_return`, `is_closed`, `is_extreme`, `onpromotion`), `earthquake` or `is_event_other` is included. `earthquake` cannot be known 16 days ahead, and `is_event_other` stays analysis only until each event type is confirmed as announced in advance. `series_age` is not a feature. Lags are at least 16 days, `transactions` is excluded and the oil price is lagged. `is_payday` is a calendar rule (the 15th and the last day of the month) based on an assumed pay schedule, not a verified payroll calendar.

**Output.** `features/subset_model.parquet` (analysis-ready table) and `features/feature_spec.json` (feature lists, categorical columns, training filter, category maps and scaling parameters), so the same transformation can be re-applied later.

**Findings.** 5,680,001 train rows are used for fitting. No category is unseen in validation or test. Compared with train, the scaled lags are about 0.06 to 0.12 standard deviations lower in validation and test, `promo_rate_observed_28` is about 0.35 to 0.38 higher and `oil_lag16` about 0.64 to 0.72 lower. The oil price in 2017 is lower than the train average but inside the range seen in train, and it can act as a proxy for time. `zero_share_28` is about 0.12 higher in validation and test, meaning that the recent history of the products sold in 2017 has more days without a sale than the train average, in line with the growth of the product range. There are 22 tree features and 15 linear features.

**Limitation.** Scaling parameters come from a subset of 10 stores and 607 items, so they must be re-fitted when the pipeline is run on all stores. Linear models still need one-hot encoding and imputation (train statistics plus missing indicators). Whether `is_payday` helps should be tested by comparing models with and without it. Tree models use the unscaled columns; the scaled `_z` columns are for linear models.

In [5]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
df = pd.read_parquet(BASE / "features" / "subset_features.parquet")
n_rows = len(df)
df["series_age"] = df["series_age"].clip(upper=365).astype("int16")

TARGET = "unit_sales"
ID_COLS = ["date", "store_nbr", "item_nbr", "split"]
# earthquake and is_event_other cannot be confirmed as known in advance: analysis only
ANALYSIS_ONLY = ["unit_sales_raw", "is_return", "is_closed", "is_extreme", "onpromotion",
                 "earthquake", "is_event_other"]
CAT_COLS = ["family", "city", "state", "store_type"]
FLAG_COLS = ["hol_national", "hol_local", "is_holiday", "is_workday_swap", "is_payday"]
INT_COLS = ["perishable", "cluster", "dow", "month"]
SALES_COLS = ["lag_16", "lag_21", "lag_28", "lag_364", "roll_mean_7", "roll_mean_28"]
OTHER_NUM = ["zero_share_28", "promo_rate_observed_28", "oil_lag16"]

# rows used to learn encodings and scaling: train split, store open
fit = df["split"].eq("train") & ~df["is_closed"]
print(f"fit rows (train, not closed): {int(fit.sum()):,} of {n_rows:,}")

# 1) categorical encoding: categories learned on train only, unseen -> -1
cat_maps = {}
for c in CAT_COLS:
    cats = sorted(df.loc[fit, c].dropna().astype(str).unique())
    cat_maps[c] = cats
    df[c + "_code"] = pd.Categorical(df[c].astype(str), categories=cats).codes.astype("int16")
    print(f"{c}: {len(cats)} categories, unseen in val/test: "
          f"{int((df.loc[~df['split'].eq('train'), c + '_code'] < 0).sum())}")

# 2) boolean flags to int8
for c in FLAG_COLS:
    df[c] = df[c].astype("int8")

# 3) scaling: log1p for sales-like columns, then z-score with train statistics
scaling = {}
for c in SALES_COLS + OTHER_NUM:
    x = df[c].astype("float64")
    use_log = c in SALES_COLS
    if use_log:
        x = np.log1p(x)
    mu, sd = float(x[fit].mean()), float(x[fit].std())
    sd = sd if sd > 0 else 1.0
    df[c + "_z"] = ((x - mu) / sd).astype("float32")
    scaling[c] = {"log1p": use_log, "mean": mu, "std": sd}

# 4) explicit feature lists
# categorical columns: integer codes are labels, not quantities
CATEGORICAL = [c + "_code" for c in CAT_COLS] + ["cluster", "dow", "month"]
tree_features = ([c + "_code" for c in CAT_COLS] + FLAG_COLS + INT_COLS
                 + SALES_COLS + ["zero_share_28", "promo_rate_observed_28", "oil_lag16"])
# linear models: numeric and 0/1 columns only; categoricals are one-hot encoded when the model is fitted
linear_features = FLAG_COLS + ["perishable"] + [c + "_z" for c in SALES_COLS + OTHER_NUM]
banned = set([TARGET] + ANALYSIS_ONLY)
assert not banned & set(tree_features + linear_features), "a leaky column is in the feature list"
assert set(CATEGORICAL) <= set(tree_features)
assert len(df) == n_rows

print("\nscaling parameters (train only):")
print(pd.DataFrame(scaling).T.round(3))
zc = [c + "_z" for c in SALES_COLS + OTHER_NUM]
print("\nmean of scaled columns by split (train ~ 0, val/test differ):")
print(df.groupby("split", observed=True)[zc[:4] + zc[-3:]].mean().round(3).T)

spec = {"target": TARGET, "id_columns": ID_COLS, "analysis_only": ANALYSIS_ONLY,
        "training_filter": "split == 'train' and is_closed == False",
        "tree_features": tree_features, "categorical_features": CATEGORICAL,
        "linear_features": linear_features,
        "notes": {"linear_models": "one-hot encode categorical_features (dow and month may be cyclical) and impute missing values with train-only statistics plus missing indicators",
                  "earthquake": "kept in the data for analysis, not a model input because it cannot be known in advance",
                  "is_event_other": "analysis only until each event type is confirmed as announced in advance",
                  "is_payday": "calendar rule (15th and last day of the month) based on an assumed pay schedule, not a verified payroll calendar; compare models with and without it"},
        "category_maps": cat_maps, "scaling": scaling, "horizon_days": 16}
with open(BASE / "features" / "feature_spec.json", "w") as f:
    json.dump(spec, f, indent=2)
df.to_parquet(BASE / "features" / "subset_model.parquet", index=False)
print(f"\nsaved subset_model.parquet: {df.shape[0]:,} rows x {df.shape[1]} cols | "
      f"{len(tree_features)} tree features, {len(linear_features)} linear features")

fit rows (train, not closed): 5,680,001 of 5,888,022
family: 30 categories, unseen in val/test: 0
city: 4 categories, unseen in val/test: 0
state: 3 categories, unseen in val/test: 0
store_type: 5 categories, unseen in val/test: 0

scaling parameters (train only):
                        log1p       mean       std
lag_16                   True   1.182544  1.131838
lag_21                   True   1.182889  1.131972
lag_28                   True   1.183761  1.132224
lag_364                  True   1.203443   1.13656
roll_mean_7              True   1.315642  1.033562
roll_mean_28             True   1.354687  1.001797
zero_share_28           False   0.346482  0.348192
promo_rate_observed_28  False      0.072  0.165748
oil_lag16               False  62.761094  24.30027

mean of scaled columns by split (train ~ 0, val/test differ):
split                     train    val   test
lag_16_z                 -0.000 -0.078 -0.103
lag_21_z                 -0.000 -0.060 -0.116
lag_28_z                

## 6. Feature selection: correlation between features

**Purpose.** Several features describe the same recent history of a series (`lag_16`, `lag_21`, `lag_28`, `roll_mean_7`, `roll_mean_28`). This step checks how strongly the numeric features are correlated and decides which columns to keep, so that the feature list is justified and not only assumed.

**Method.**
- Only fitting rows are used (train split, store open), the same rows used to learn encodings and scaling in section 5, so validation and test do not influence the decision.
- Sales-like columns are transformed with `log1p`, as in section 5. Pearson correlation is computed on pairwise complete observations (missing values are not filled).
- Pairs with an absolute correlation of at least 0.90 are listed. Each feature is also correlated with `log1p(unit_sales)` to see which member of a correlated group carries the most signal. Missing shares are printed because a feature with many missing values is costlier to keep.

**Output.** The correlation matrix, the list of highly correlated pairs, the correlation of each feature with the target and the missing share of each feature.

**Limitation.** Pearson correlation only captures linear relations and uses the training period only. Tree models are not harmed by correlated features, so a column is removed from `linear_features` only when the evidence supports it, and tree features are kept unless a column adds nothing. The 0.90 cut-off is a convention, not a rule of the data.

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
THRESHOLD = 0.90          # |r| at or above this counts as "highly correlated"

SALES_COLS = ["lag_16", "lag_21", "lag_28", "lag_364", "roll_mean_7", "roll_mean_28"]
OTHER_NUM = ["zero_share_28", "promo_rate_observed_28", "oil_lag16"]
cols = ["split", "is_closed", "unit_sales"] + SALES_COLS + OTHER_NUM
df = pd.read_parquet(BASE / "features" / "subset_model.parquet", columns=cols)

fit = df[df["split"].eq("train") & ~df["is_closed"]]      # same rows as the encoding and scaling fit
X = fit[SALES_COLS + OTHER_NUM].astype("float64").copy()
X[SALES_COLS] = np.log1p(X[SALES_COLS])                   # sales-like columns on the log scale, as in section 5

corr = X.corr()                                           # Pearson, pairwise complete observations
print("correlation matrix (train, store open, log1p for sales columns):")
print(corr.round(2).to_string())

pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
             .stack().rename("r").reset_index()
             .rename(columns={"level_0": "feature_a", "level_1": "feature_b"}))
high = pairs[pairs["r"].abs() >= THRESHOLD].sort_values("r", key=abs, ascending=False)
print(f"\npairs with |r| >= {THRESHOLD}:")
print(high.round(3).to_string(index=False) if len(high) else "none")

target = np.log1p(fit["unit_sales"].astype("float64"))
with_target = X.corrwith(target).rename("corr_with_log1p_sales").round(3)
print("\ncorrelation of each feature with log1p(unit_sales):")
print(with_target.sort_values(ascending=False).to_string())
print("\nmissing share of each feature in the fit rows:")
print(X.isna().mean().round(3).to_string())

correlation matrix (train, store open, log1p for sales columns):
                        lag_16  lag_21  lag_28  lag_364  roll_mean_7  roll_mean_28  zero_share_28  promo_rate_observed_28  oil_lag16
lag_16                    1.00    0.72    0.70     0.58         0.87          0.83          -0.69                    0.07       0.04
lag_21                    0.72    1.00    0.74     0.61         0.88          0.84          -0.70                    0.08       0.04
lag_28                    0.70    0.74    1.00     0.61         0.80          0.85          -0.71                    0.09       0.04
lag_364                   0.58    0.61    0.61     1.00         0.66          0.68          -0.54                    0.03       0.06
roll_mean_7               0.87    0.88    0.80     0.66         1.00          0.95          -0.80                    0.09       0.04
roll_mean_28              0.83    0.84    0.85     0.68         0.95          1.00          -0.82                    0.11       0.05
zero

**Findings.** Only one pair of features reaches the 0.90 cut-off: `roll_mean_7` and `roll_mean_28` (r = 0.949). The other sales-based features are correlated at 0.58 to 0.88, so each keeps some separate information; `zero_share_28` is negatively correlated with the rolling means (about -0.8) but measures how often a product sells, not how much. `roll_mean_28` (0.772) and `roll_mean_7` (0.765) have the highest correlation with log sales, `lag_364` the lowest among the sales features (0.612). `promo_rate_observed_28` and `oil_lag16` are almost uncorrelated with log sales (0.04) and `promo_rate_observed_28` is missing for 45.9% of the fit rows.

**Decision.** `roll_mean_7_z` is removed from `linear_features` because it duplicates `roll_mean_28_z`, which is more stable and slightly more correlated with the target; this lowers the number of linear features from 15 to 14. `tree_features` stay unchanged (22) because tree models are not harmed by correlated inputs. `lag_364` is kept despite 30% missing values because it is the only feature that looks a full year back. `oil_lag16` is flagged for review next semester, since a weak link with sales and a shifted distribution in 2017 suggest it may only act as a proxy for time. A weak linear correlation does not remove `promo_rate_observed_28`: promotions are rare, and the EDA shows a clear difference in sales on promotion days.

**Limitation.** Correlation is computed on the 10-store subset and only measures linear association; it does not show how much a feature improves a model. Feature importance from the models of the next semester should confirm or reverse these choices.

In [7]:
import json
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
path = BASE / "features" / "feature_spec.json"
spec = json.load(open(path))

DROP_LINEAR = {"roll_mean_7_z": "r = 0.95 with roll_mean_28; 28-day mean kept (higher correlation with the target, less noisy)"}

before = len(spec["linear_features"])
spec["linear_features"] = [c for c in spec["linear_features"] if c not in DROP_LINEAR]
spec["feature_selection"] = {
    "method": "Pearson correlation on log1p scale, train rows with store open, threshold |r| >= 0.90",
    "dropped_from_linear_features": DROP_LINEAR,
    "tree_features": "unchanged: correlated features do not harm tree models",
    "flagged_for_review": {"oil_lag16": "weak link with sales and may act as a proxy for time; check importance next semester",
                           "zero_share_28": "r about -0.8 with the rolling means but measures intermittency; kept"},
}
json.dump(spec, open(path, "w"), indent=2)

print(f"linear features: {before} -> {len(spec['linear_features'])} | tree features: {len(spec['tree_features'])} (unchanged)")
print("dropped:", list(DROP_LINEAR))

linear features: 15 -> 14 | tree features: 22 (unchanged)
dropped: ['roll_mean_7_z']
